# Обучение DenseNet-121 на GTSRB

Этот блокнот демонстрирует обучение DenseNet-121 на датасете GTSRB.

DenseNet использует плотные связи между слоями, где каждый слой получает
входные данные от всех предыдущих слоев.

In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from configs.densenet_config import DenseNetConfig
from src.data_loader_densenet import get_densenet_data_loaders
from src.model_densenet import create_densenet121_gpu
from src.train_densenet import DenseNetTrainer, evaluate_densenet
from src.gpu_utils import GPUManager, profile_model_performance



In [ ]:
# 1. Настройка GPU
gpu_manager = GPUManager()
device = gpu_manager.device
gpu_manager.print_gpu_info()

In [ ]:
# 2. Загрузка данных
config = DenseNetConfig
train_loader, val_loader, test_loader = get_densenet_data_loaders(config, device)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Classes: {config.NUM_CLASSES}")

In [ ]:
# 3. Проверка батча
def show_batch(loader, class_names, n=5):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(class_names[labels[i].item()])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

class_names = config.get_class_names()
print("Training batch:")
show_batch(train_loader, class_names)

In [ ]:
# 4. Создание модели DenseNet-121
model = create_densenet121_gpu(config, device)

In [ ]:
# 5. Обучение DenseNet-121
trainer = DenseNetTrainer(
    model, 
    config, 
    device, 
    use_amp=True,
    gradient_accumulation_steps=1
)

history = trainer.train(train_loader, val_loader)

In [ ]:
# 6. Визуализация обучения
trainer.plot_history()

In [ ]:
# 7. Оценка на тестовых данных
results, preds, labels = evaluate_densenet(model, test_loader, config, device, use_amp=True)

print("\nTest Results:")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1-Score: {results['f1_score']:.4f}")

In [ ]:
# 8. Профилирование производительности
perf_stats = profile_model_performance(model, device)
print(f"\nInference Performance:")
print(f"Average time: {perf_stats['avg_inference_time_ms']:.2f} ms")
print(f"FPS: {perf_stats['fps']:.2f}")

In [ ]:
# 9. Сравнение всех 5 архитектур
print("\n" + "=" * 70)
print("COMPLETE MODEL COMPARISON (5 ARCHITECTURES)")
print("=" * 70)

models_comparison = {
    'ResNet-50': {'accuracy': 0.985, 'size_mb': 98, 'time_ms': 20, 'params': '25.6M'},
    'EfficientNet-B0': {'accuracy': 0.977, 'size_mb': 20, 'time_ms': 10, 'params': '5.3M'},
    'MobileNetV3-Large': {'accuracy': 0.972, 'size_mb': 21, 'time_ms': 5, 'params': '5.5M'},
    'ViT-B/16': {'accuracy': 0.978, 'size_mb': 330, 'time_ms': 15, 'params': '86M'},
    'DenseNet-121': {'accuracy': results['accuracy'], 'size_mb': model.get_model_size_mb(), 
                    'time_ms': perf_stats['avg_inference_time_ms'], 'params': f"{model.get_total_params()/1e6:.1f}M"}
}

for name, metrics in models_comparison.items():
    print(f"\n{name}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Size: {metrics['size_mb']:.1f} MB")
    print(f"  Inference: {metrics['time_ms']:.1f} ms")
    print(f"  Parameters: {metrics['params']}")

In [ ]:
# 10. Визуализация плотных связей DenseNet
def visualize_dense_connections(model):
    """
    Визуализация плотных связей в DenseNet
    """
    print("\nDenseNet-121 Architecture Analysis:")
    print("-" * 40)
    
    feature_info = model.get_feature_map_size()
    total_layers = sum(info['layers'] for info in feature_info.values())
    
    print(f"Total dense blocks: 4")
    print(f"Total dense layers: {total_layers}")
    print("\nDense blocks configuration:")
    
    for block_name, info in feature_info.items():
        connections = info['layers'] * (info['layers'] + 1) // 2
        print(f"  {block_name}: {info['layers']} layers, {connections} dense connections")
    
    print(f"\nGrowth rate: 32")
    print(f"Total feature channels at final layer: 1024")

visualize_dense_connections(model)

In [ ]:
# 11. Анализ ошибок по классам
def analyze_class_errors(preds, labels, class_names):
    """Подробный анализ ошибок по классам"""
    class_errors = {}
    total_per_class = {}
    
    for pred, true in zip(preds, labels):
        if true not in total_per_class:
            total_per_class[true] = 0
            class_errors[true] = 0
        total_per_class[true] += 1
        if pred != true:
            class_errors[true] += 1
    
    print("\nClass-wise Error Analysis:")
    print("-" * 50)
    print("Class Name | Total | Errors | Error Rate")
    print("-" * 50)
    
    error_rates = {}
    for class_id in sorted(class_errors.keys()):
        total = total_per_class[class_id]
        errors = class_errors[class_id]
        error_rate = errors / total
        error_rates[class_id] = error_rate
        print(f"{class_names[class_id]:<20} | {total:5d} | {errors:6d} | {error_rate:.2%}")
    
    # Классы с наибольшей ошибкой
    worst_classes = sorted(error_rates.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nClasses with highest error rate:")
    for class_id, rate in worst_classes:
        print(f"  {class_names[class_id]}: {rate:.2%}")

analyze_class_errors(preds, labels, class_names)

In [ ]:
# 12. Сохранение модели
torch.save(model.state_dict(), config.RUNS_PATH / 'final_model.pth')
print(f"\nModel saved to: {config.RUNS_PATH / 'final_model.pth'}")